### 사용자 행동 루프 문제 구간 탐색

#### 분석 목적
- 사용자 행동 루프의 5개 영역을 빠르게 진단하여, 문제 규모와 다음 행동 전환 정도를 기준으로 개선이 필요한 핵심 구간 2개 선정

#### 분석 영역
- 친구 연결
- 질문 참여·콘텐츠
- 투표 수신·반응
- 상점·첫 구매
- 구매 후 이용·재구매

#### 데이터 불러오기

In [95]:
import pandas as pd
import numpy as np
import os

# 파일이 있는 기본 폴더 경로 설정
base_path = r"C:\DA15_Part4\sns-project\team4-final-project\data\interim"

# 이미지에 있는 파일명들을 각각의 데이터프레임 변수로 로드
df_friendrequest = pd.read_csv(os.path.join(base_path, "accounts_friendrequest_preprocessed.csv"))
df_paymenthistory = pd.read_csv(os.path.join(base_path, "accounts_paymenthistory_preprocessed.csv"))
df_user_contacts = pd.read_csv(os.path.join(base_path, "accounts_user_contacts_preprocessed.csv"))
df_user = pd.read_csv(os.path.join(base_path, "accounts_user_preprocessed.csv"))
df_userquestionrecord = pd.read_csv(os.path.join(base_path, "accounts_userquestionrecord_preprocessed.csv"))
df_hackle_events = pd.read_csv(os.path.join(base_path, "hackle_events_preprocessed.csv"))
df_hackle_properties = pd.read_csv(os.path.join(base_path, "hackle_properties_preprocessed.csv"))
df_polls_question = pd.read_csv(os.path.join(base_path, "polls_question_preprocessed.csv"))
df_polls_questionpiece = pd.read_csv(os.path.join(base_path, "polls_questionpiece_preprocessed.csv"))
df_polls_questionset = pd.read_csv(os.path.join(base_path, "polls_questionset_preprocessed.csv"))
df_pointhistory = pd.read_csv(os.path.join(base_path, "accounts_pointhistory_preprocessed.csv"))

In [96]:
dfs = {
    "friendrequest": df_friendrequest,
    "paymenthistory": df_paymenthistory,
    "user_contacts": df_user_contacts,
    "user": df_user,
    "userquestionrecord": df_userquestionrecord,
    "hackle_events": df_hackle_events,
    "hackle_properties": df_hackle_properties,
    "polls_question": df_polls_question,
    "polls_questionpiece": df_polls_questionpiece,
    "polls_questionset": df_polls_questionset,
    "pointhistory": df_pointhistory
}

for name, df in dfs.items():
    print(f"{name:25s} {df.shape}")

friendrequest             (17147175, 7)
paymenthistory            (95140, 5)
user_contacts             (5063, 4)
user                      (677080, 11)
userquestionrecord        (1217558, 12)
hackle_events             (11441319, 10)
hackle_properties         (327431, 8)
polls_question            (5025, 3)
polls_questionpiece       (1264349, 5)
polls_questionset         (157705, 6)
pointhistory              (2338918, 5)


#### 1. 친구 연결 → 질문 참여

##### 핵심 질문
- 친구 관계가 실제 질문 참여로 이어지는가?
- 친구가 없는 사용자가 많고, 해당 사용자의 질문 참여율까지 낮다면 초기 친구 연결 과정이 주요 이탈 구간일 가능성이 있다.

In [3]:
# 수락된 친구 관계 추출
df_accepted = (
    df_friendrequest.loc[
        df_friendrequest["status"] == "A",
        ["send_user_id", "receive_user_id"]
    ]
    .dropna()
    .drop_duplicates()
)

df_accepted.head()

,send_user_id,receive_user_id
1,837521,832151
2,837521,832340
3,837521,833041
7,837523,835871
8,837524,835869


In [4]:
# 친구 관계를 양방향으로 변환
df_friend_edges = pd.concat([
    df_accepted.rename(columns={
        "send_user_id": "user_id",
        "receive_user_id": "friend_id"
    }),
    
    df_accepted.rename(columns={
        "receive_user_id": "user_id",
        "send_user_id": "friend_id"
    })
], ignore_index=True)

df_friend_edges = df_friend_edges.drop_duplicates()

In [5]:
# 사용자별 친구 수 계산
df_friend_count = (
    df_friend_edges
    .groupby("user_id")["friend_id"]
    .nunique()
    .reset_index(name="friend_count")
)

df_friend_count.head()

,user_id,friend_count
0,831962,1
1,832151,8
2,832340,11
3,833024,36
4,833041,29


In [6]:
# 사용자별 질문 참여 횟수 계산
df_participation = (
    df_userquestionrecord
    .groupby("user_id")
    .size()
    .reset_index(name="participation_count")
)

df_participation.head()

,user_id,participation_count
0,838023,22
1,838466,131
2,838642,10
3,839357,212
4,840293,170


In [7]:
# 전체 사용자 기준 결합
df_friend_analysis = (
    df_user[["id"]]
    .rename(columns={"id": "user_id"})
    .merge(df_friend_count, on="user_id", how="left")
    .merge(df_participation, on="user_id", how="left")
)

df_friend_analysis[
    ["friend_count", "participation_count"]
] = df_friend_analysis[
    ["friend_count", "participation_count"]
].fillna(0)

In [8]:
# 친구 수 그룹 설정
df_friend_analysis["participated"] = (
    df_friend_analysis["participation_count"] > 0
)

df_friend_analysis["friend_group"] = pd.cut(
    df_friend_analysis["friend_count"],
    bins=[-1, 0, 2, 5, np.inf],
    labels=["0명", "1~2명", "3~5명", "6명+"]
)

In [9]:
# 친구 수별 질문 참여율
df_friend_result = (
    df_friend_analysis
    .groupby("friend_group", observed=True)
    .agg(
        user_count=("user_id", "nunique"),
        participation_rate=("participated", "mean"),
        avg_participation=("participation_count", "mean")
    )
    .reset_index()
)

df_friend_result

,friend_group,user_count,participation_rate,avg_participation
0,0명,26801,0.000709,0.052797
1,1~2명,18213,0.002032,0.249931
2,3~5명,24219,0.002932,0.196581
3,6명+,607847,0.007768,1.985417


In [10]:
# 핵심 수치
no_friend_rate = (
    df_friend_analysis["friend_count"] == 0
).mean()

friend0_rate = df_friend_analysis.loc[
    df_friend_analysis["friend_count"] == 0,
    "participated"
].mean()

friend3_rate = df_friend_analysis.loc[
    df_friend_analysis["friend_count"] >= 3,
    "participated"
].mean()

friend_participation_gap = friend3_rate - friend0_rate

print(f"친구 0명 사용자 비율: {no_friend_rate:.1%}")
print(f"친구 0명 질문 참여율: {friend0_rate:.1%}")
print(f"친구 3명 이상 질문 참여율: {friend3_rate:.1%}")
print(f"참여율 차이: {friend_participation_gap * 100:.1f}%p")

친구 0명 사용자 비율: 4.0%
친구 0명 질문 참여율: 0.1%
친구 3명 이상 질문 참여율: 0.8%
참여율 차이: 0.7%p


#### 2. 질문 참여·콘텐츠

##### 핵심 질문
- 반복되는 질문 경험이 스킵 증가로 이어지는가?
- 동일 질문을 여러 번 경험할수록 스킵률이 높아진다면 콘텐츠 반복 노출이 참여 중단의 원인일 가능성이 있다.

In [11]:
# 질문 이벤트 추출
df_question_events = df_hackle_events.loc[
    df_hackle_events["event_key"].isin(
        ["skip_question", "complete_question"]
    ),
    ["event_datetime", "event_key", "session_id", "question_id"]
].copy()

In [12]:
# 사용자 ID 연결
# hackle_properties에서 세션별 user_id 추출
df_hackle_user = (
    df_hackle_properties[
        ["session_id", "user_id"]
    ]
    .dropna(subset=["session_id", "user_id"])
    .drop_duplicates(subset="session_id")
)

# 질문 이벤트에 user_id 연결
df_question_events = (
    df_question_events
    .merge(
        df_hackle_user,
        on="session_id",
        how="left"
    )
    .dropna(subset=["user_id", "question_id"])
)

# user_id 타입 정리
df_question_events["user_id"] = (
    df_question_events["user_id"].astype("int64")
)

df_question_events.head()

,event_datetime,event_key,session_id,question_id,user_id
0,2023-07-18 15:41:11,skip_question,V3LLobUFnpeakqUYgsVUnaWEFoe2,817.0,1159545
1,2023-08-09 16:04:30,skip_question,65e6274a-bf54-4638-83df-f37b6e6ab31c,4863.0,1188691
2,2023-08-06 19:50:21,skip_question,KqT1MV9g8OhVkpaA2Jj8wrxwHO93,1194.0,1496697
3,2023-07-26 22:30:04,skip_question,7f0fa1b4-41b8-4267-9033-97c741099f10,4189.0,1557770
5,2023-08-07 18:07:45,skip_question,6eDbebJ76BT7AYauychTCvu3Vkl1,804.0,1037894


In [13]:
# 동일 질문 경험 순서 계산
df_question_events = df_question_events.sort_values(
    ["user_id", "question_id", "event_datetime"]
)

df_question_events["exposure_order"] = (
    df_question_events
    .groupby(["user_id", "question_id"])
    .cumcount() + 1
)

In [14]:
# 반복 노출 그룹 생성
df_question_events["exposure_group"] = np.select(
    [
        df_question_events["exposure_order"] == 1,
        df_question_events["exposure_order"] == 2
    ],
    [
        "1회",
        "2회"
    ],
    default="3회+"
)

df_question_events["is_skip"] = (
    df_question_events["event_key"] == "skip_question"
)

In [15]:
# 반복 횟수별 스킵률
df_content_result = (
    df_question_events
    .groupby("exposure_group")
    .agg(
        event_count=("event_key", "size"),
        skip_rate=("is_skip", "mean")
    )
    .reindex(["1회", "2회", "3회+"])
)

df_content_result

,event_count,skip_rate
exposure_group,,
1회,427586,1.0
2회,12668,1.0
3회+,1661,1.0


In [16]:
# 핵심 수치
overall_skip_rate = df_question_events["is_skip"].mean()

first_skip_rate = df_question_events.loc[
    df_question_events["exposure_order"] == 1,
    "is_skip"
].mean()

repeat_skip_rate = df_question_events.loc[
    df_question_events["exposure_order"] >= 2,
    "is_skip"
].mean()

repeat_skip_gap = repeat_skip_rate - first_skip_rate

print(f"전체 질문 스킵률: {overall_skip_rate:.1%}")
print(f"첫 경험 스킵률: {first_skip_rate:.1%}")
print(f"반복 경험 스킵률: {repeat_skip_rate:.1%}")
print(f"반복 시 스킵률 변화: {repeat_skip_gap * 100:.1f}%p")

전체 질문 스킵률: 100.0%
첫 경험 스킵률: 100.0%
반복 경험 스킵률: 100.0%
반복 시 스킵률 변화: 0.0%p


#### 3. 투표 수신 → 반응

##### 핵심 질문
- 받은 투표가 실제 확인 행동으로 이어지는가?
- 투표를 받고도 확인하지 않는 사용자가 많다면 서비스의 핵심 보상 경험에서 이탈이 발생하고 있을 가능성이 있다.

In [17]:
# Hackle 분석 기간 확인
hackle_start = df_hackle_events["event_datetime"].min()
hackle_end = df_hackle_events["event_datetime"].max()

print("Hackle 시작일:", hackle_start)
print("Hackle 종료일:", hackle_end)

Hackle 시작일: 2023-07-18 00:00:00
Hackle 종료일: 2023-08-10 23:59:59


In [18]:
# 같은 기간의 투표 기록 추출
df_vote_period = df_userquestionrecord.loc[
    df_userquestionrecord["created_at"].between(
        hackle_start, hackle_end
    )
].dropna(subset=["chosen_user_id"]).copy()

df_vote_period["chosen_user_id"] = (
    df_vote_period["chosen_user_id"].astype("int64")
)

In [19]:
# 사용자별 받은 투표와 확인 수
df_received_vote = (
    df_vote_period
    .groupby("chosen_user_id")
    .agg(
        received_votes=("id", "size"),
        read_votes=("has_read", "sum")
    )
    .reset_index()
    .rename(columns={"chosen_user_id": "user_id"})
)

df_received_vote.head()

,user_id,received_votes,read_votes
0,833203,1,1
1,837549,2,0
2,837605,1,0
3,839494,1,1
4,840293,1,1


In [20]:
# 확인율 계산
df_received_vote["read_rate"] = (
    df_received_vote["read_votes"]
    / df_received_vote["received_votes"]
)

df_received_vote["read_group"] = np.where(
    df_received_vote["read_votes"] > 0,
    "1회 이상 확인",
    "미확인"
)

vote_read_rate = (
    df_received_vote["read_votes"].sum()
    / df_received_vote["received_votes"].sum()
)

print(f"받은 투표 확인율: {vote_read_rate:.1%}")
print(f"받은 투표 미확인율: {1 - vote_read_rate:.1%}")

받은 투표 확인율: 24.3%
받은 투표 미확인율: 75.7%


In [21]:
# 상점 방문 사용자 추출

# session_id와 user_id 연결용 데이터 생성
df_hackle_user = (
    df_hackle_properties[
        ["session_id", "user_id"]
    ]
    .dropna(subset=["session_id", "user_id"])
    .drop_duplicates(subset="session_id")
    .copy()
)

df_hackle_user["user_id"] = (
    df_hackle_user["user_id"].astype("int64")
)

# 상점 방문 이벤트 추출
df_shop_session = df_hackle_events.loc[
    df_hackle_events["event_key"] == "view_shop",
    ["session_id"]
].copy()

# 상점 방문 사용자 연결
df_shop_user = (
    df_shop_session
    .merge(
        df_hackle_user,
        on="session_id",
        how="left"
    )
    .dropna(subset=["user_id"])
)

shop_users = set(
    df_shop_user["user_id"].astype("int64")
)

print(f"상점 방문 사용자 수: {len(shop_users):,}명")

상점 방문 사용자 수: 17,122명


In [22]:
# 투표 확인 여부와 상점 방문 비교
df_received_vote["shop_visited"] = (
    df_received_vote["user_id"].isin(shop_users)
)

df_vote_result = (
    df_received_vote
    .groupby("read_group")
    .agg(
        user_count=("user_id", "nunique"),
        avg_received_votes=("received_votes", "mean"),
        shop_visit_rate=("shop_visited", "mean")
    )
)

df_vote_result

,user_count,avg_received_votes,shop_visit_rate
read_group,,,
1회 이상 확인,435,1.866667,0.091954
미확인,1173,1.708440,0.012788


In [23]:
# 핵심 수치
vote_shop_rates = (
    df_received_vote
    .groupby("read_group")["shop_visited"]
    .mean()
)

checked_shop_rate = vote_shop_rates.get(
    "1회 이상 확인",
    np.nan
)

unchecked_shop_rate = vote_shop_rates.get(
    "미확인",
    np.nan
)

vote_shop_gap = (
    checked_shop_rate - unchecked_shop_rate
)

print(f"투표 확인 사용자 상점 방문율: {checked_shop_rate:.1%}")
print(f"투표 미확인 사용자 상점 방문율: {unchecked_shop_rate:.1%}")
print(f"상점 방문율 차이: {vote_shop_gap * 100:.1f}%p")

투표 확인 사용자 상점 방문율: 9.2%
투표 미확인 사용자 상점 방문율: 1.3%
상점 방문율 차이: 7.9%p


#### 4. 상점 → 첫 구매

##### 핵심 질문
- 상점에 방문한 사용자가 실제 구매까지 이어지는가?
- 상점 방문 후 구매하지 않는 사용자가 많고, 특히 반복 방문 후에도 미구매인 사용자가 많다면 결제 직전 단계에 전환 문제가 있다고 판단할 수 있다.

In [24]:
# 상점 및 구매 이벤트 추출
df_shop_events = df_hackle_events.loc[
    df_hackle_events["event_key"].isin(
        ["view_shop", "complete_purchase"]
    ),
    ["event_datetime", "event_key", "session_id"]
].copy()

df_shop_events.head()

,event_datetime,event_key,session_id
3,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2
879,2023-07-22 00:26:48,view_shop,SH9zv4LwFNYIEjYKk9sluZpqEJx1
1427,2023-07-20 15:53:09,view_shop,nQW8XfTA5vg4IxglIFOaA4ut9gL2
1586,2023-08-06 20:42:21,complete_purchase,DFD8537F-41C0-4809-8864-49AEDF77BAB2
1656,2023-07-25 01:49:09,view_shop,b769704d-f5d2-4b2a-898b-8c48c9796459


In [25]:
# 사용자 ID 연결
df_shop_events = (
    df_shop_events
    .merge(
        df_hackle_user,
        on="session_id",
        how="left"
    )
    .dropna(subset=["user_id"])
)

df_shop_events["user_id"] = (
    df_shop_events["user_id"].astype("int64")
)

df_shop_events.head()

,event_datetime,event_key,session_id,user_id
0,2023-08-05 01:46:10,view_shop,XVYNT6zfhFWqIg9omwg2AHDjTLx2,1224793
1,2023-07-22 00:26:48,view_shop,SH9zv4LwFNYIEjYKk9sluZpqEJx1,1289482
2,2023-07-20 15:53:09,view_shop,nQW8XfTA5vg4IxglIFOaA4ut9gL2,1050604
3,2023-08-06 20:42:21,complete_purchase,DFD8537F-41C0-4809-8864-49AEDF77BAB2,1542167
4,2023-07-25 01:49:09,view_shop,b769704d-f5d2-4b2a-898b-8c48c9796459,1084334


In [26]:
# 사용자별 상점 방문 횟수
df_shop_visit = (
    df_shop_events.loc[
        df_shop_events["event_key"] == "view_shop"
    ]
    .groupby("user_id")
    .agg(
        shop_visit_count=("event_key", "size"),
        first_shop_at=("event_datetime", "min")
    )
    .reset_index()
)

df_shop_visit.head()

,user_id,shop_visit_count,first_shop_at
0,836055,1,2023-07-19 17:52:17
1,836487,1,2023-07-26 23:57:55
2,836498,1,2023-07-21 01:54:29
3,837538,1,2023-07-18 23:34:53
4,837605,1,2023-07-20 08:39:41


In [27]:
# 사용자별 첫 구매 시점
df_first_purchase = (
    df_shop_events.loc[
        df_shop_events["event_key"] == "complete_purchase"
    ]
    .groupby("user_id")
    .agg(
        first_purchase_at=("event_datetime", "min")
    )
    .reset_index()
)

df_first_purchase.head()

,user_id,first_purchase_at
0,838673,2023-07-21 09:27:31
1,843573,2023-07-23 12:32:48
2,845840,2023-08-02 09:23:08
3,846936,2023-07-23 01:19:24
4,848862,2023-08-03 16:54:32


In [28]:
# 상점 방문과 구매 연결
df_shop_analysis = df_shop_visit.merge(
    df_first_purchase,
    on="user_id",
    how="left"
)

df_shop_analysis["converted"] = (
    df_shop_analysis["first_purchase_at"].notna()
    & (
        df_shop_analysis["first_purchase_at"]
        >= df_shop_analysis["first_shop_at"]
    )
)

df_shop_analysis.head()

,user_id,shop_visit_count,first_shop_at,first_purchase_at,converted
0,836055,1,2023-07-19 17:52:17,NaN,False
1,836487,1,2023-07-26 23:57:55,NaN,False
2,836498,1,2023-07-21 01:54:29,NaN,False
3,837538,1,2023-07-18 23:34:53,NaN,False
4,837605,1,2023-07-20 08:39:41,NaN,False


In [29]:
# 반복 방문 미구매자 정의
df_shop_analysis["repeat_no_purchase"] = (
    (df_shop_analysis["shop_visit_count"] >= 2)
    & (~df_shop_analysis["converted"])
)

df_shop_analysis[
    ["user_id", "shop_visit_count", "converted", "repeat_no_purchase"]
].head()

,user_id,shop_visit_count,converted,repeat_no_purchase
0,836055,1,False,False
1,836487,1,False,False
2,836498,1,False,False
3,837538,1,False,False
4,837605,1,False,False


In [30]:
# 핵심 수치
shop_conversion_rate = (
    df_shop_analysis["converted"].mean()
)

shop_no_purchase_rate = (
    1 - shop_conversion_rate
)

repeat_no_purchase_count = (
    df_shop_analysis["repeat_no_purchase"].sum()
)

repeat_no_purchase_rate = (
    df_shop_analysis["repeat_no_purchase"].mean()
)

print(f"상점 방문 사용자: {len(df_shop_analysis):,}명")
print(f"상점 → 구매 전환율: {shop_conversion_rate:.1%}")
print(f"상점 미구매율: {shop_no_purchase_rate:.1%}")

print(
    f"2회 이상 방문 후 미구매: "
    f"{repeat_no_purchase_count:,}명 "
    f"({repeat_no_purchase_rate:.1%})"
)

상점 방문 사용자: 17,122명
상점 → 구매 전환율: 9.4%
상점 미구매율: 90.6%
2회 이상 방문 후 미구매: 3,776명 (22.1%)


In [31]:
# 상점 방문 횟수 그룹 생성
df_shop_analysis["shop_group"] = pd.cut(
    df_shop_analysis["shop_visit_count"],
    bins=[0, 1, 3, np.inf],
    labels=[
        "1회",
        "2~3회",
        "4회+"
    ]
)

In [32]:
# 방문 횟수별 구매 전환율
df_shop_result = (
    df_shop_analysis
    .groupby("shop_group", observed=True)
    .agg(
        user_count=("user_id", "nunique"),
        conversion_rate=("converted", "mean")
    )
)

df_shop_result

,user_count,conversion_rate
shop_group,,
1회,12224,0.040494
2~3회,3995,0.191740
4회+,903,0.394241


In [33]:
# 방문 횟수별 전환율 핵심 비교
shop_1_rate = df_shop_analysis.loc[
    df_shop_analysis["shop_group"] == "1회",
    "converted"
].mean()

shop_repeat_rate = df_shop_analysis.loc[
    df_shop_analysis["shop_visit_count"] >= 2,
    "converted"
].mean()

shop_conversion_gap = (
    shop_repeat_rate - shop_1_rate
)

print(f"상점 1회 방문자 구매율: {shop_1_rate:.1%}")
print(f"상점 2회 이상 방문자 구매율: {shop_repeat_rate:.1%}")
print(f"구매율 차이: {shop_conversion_gap * 100:.1f}%p")

상점 1회 방문자 구매율: 4.0%
상점 2회 이상 방문자 구매율: 22.9%
구매율 차이: 18.9%p


#### 5. 첫 구매 → 재구매

##### 핵심 질문
- 첫 구매 이후 실제 반복 구매로 이어지는가?
- 첫 구매 후 다시 구매하지 않는 사용자가 많다면 구매 이후 이용 및 재구매 과정에 개선이 필요할 가능성이 있다.

In [34]:
# 결제 데이터 날짜형 변환 및 정렬
df_purchase = (
    df_paymenthistory
    .dropna(subset=["user_id", "created_at"])
    .copy()
)

# created_at을 확실하게 datetime으로 변환
df_purchase["created_at"] = (
    pd.to_datetime(
        df_purchase["created_at"],
        errors="coerce",
        utc=True
    )
    .dt.tz_convert(None)
)

# 날짜 변환에 실패한 행 제거
df_purchase = (
    df_purchase
    .dropna(subset=["created_at"])
    .sort_values(["user_id", "created_at"])
    .copy()
)

# 사용자별 구매 순서
df_purchase["purchase_order"] = (
    df_purchase
    .groupby("user_id")
    .cumcount() + 1
)

df_purchase.head()

,id,productId,phone_type,created_at,user_id,purchase_order
5757,5919,heart.4000,I,2023-05-14 07:59:46,833041,1
13371,13794,heart.4000,I,2023-05-15 01:50:06,833041,2
49788,51467,heart.777,I,2023-05-21 23:55:46,834112,1
88213,91075,heart.777,I,2023-06-10 14:17:45,835815,1
88215,91077,heart.200,I,2023-06-10 14:19:02,835815,2


In [35]:
# 첫 구매 추출
df_first = (
    df_purchase.loc[
        df_purchase["purchase_order"] == 1,
        ["user_id", "created_at"]
    ]
    .rename(
        columns={
            "created_at": "first_purchase"
        }
    )
)

df_first.head()

,user_id,first_purchase
5757,833041,2023-05-14 07:59:46
49788,834112,2023-05-21 23:55:46
88213,835815,2023-06-10 14:17:45
19510,835839,2023-05-15 16:37:30
86747,835888,2023-06-06 04:58:49


In [36]:
# 두 번째 구매 추출
df_second = (
    df_purchase.loc[
        df_purchase["purchase_order"] == 2,
        ["user_id", "created_at"]
    ]
    .rename(
        columns={
            "created_at": "second_purchase"
        }
    )
)

df_second.head()

,user_id,second_purchase
13371,833041,2023-05-15 01:50:06
88215,835815,2023-06-10 14:19:02
86748,835888,2023-06-06 04:59:22
50229,838054,2023-05-22 06:31:23
69102,838181,2023-05-27 00:28:47


In [37]:
# 첫 구매와 두 번째 구매 연결
df_repurchase = df_first.merge(
    df_second,
    on="user_id",
    how="left"
)

df_repurchase["repurchased"] = (
    df_repurchase["second_purchase"].notna()
)

df_repurchase.head()

,user_id,first_purchase,second_purchase,repurchased
0,833041,2023-05-14 07:59:46,2023-05-15 01:50:06,True
1,834112,2023-05-21 23:55:46,NaT,False
2,835815,2023-06-10 14:17:45,2023-06-10 14:19:02,True
3,835839,2023-05-15 16:37:30,NaT,False
4,835888,2023-06-06 04:58:49,2023-06-06 04:59:22,True


In [38]:
# 재구매까지 걸린 시간
df_repurchase["days_to_second"] = (
    df_repurchase["second_purchase"]
    - df_repurchase["first_purchase"]
).dt.total_seconds() / 86400

df_repurchase[
    ["user_id", "first_purchase", "second_purchase", "days_to_second"]
].head()

,user_id,first_purchase,second_purchase,days_to_second
0,833041,2023-05-14 07:59:46,2023-05-15 01:50:06,0.743287
1,834112,2023-05-21 23:55:46,NaT,NaN
2,835815,2023-06-10 14:17:45,2023-06-10 14:19:02,0.000891
3,835839,2023-05-15 16:37:30,NaT,NaN
4,835888,2023-06-06 04:58:49,2023-06-06 04:59:22,0.000382


In [39]:
# 전체 기간 재구매율
repurchase_rate = (
    df_repurchase["repurchased"].mean()
)

print(f"전체 구매자 수: {len(df_repurchase):,}명")
print(f"전체 기간 재구매율: {repurchase_rate:.1%}")
print(f"전체 기간 미재구매율: {1 - repurchase_rate:.1%}")

전체 구매자 수: 59,192명
전체 기간 재구매율: 27.3%
전체 기간 미재구매율: 72.7%


In [40]:
# 30일 이상 관찰 가능한 사용자
payment_end = df_purchase["created_at"].max()

df_repurchase["eligible_30d"] = (
    df_repurchase["first_purchase"]
    <= payment_end - pd.Timedelta(days=30)
)

eligible_30d_count = (
    df_repurchase["eligible_30d"].sum()
)

print(f"결제 데이터 마지막 날짜: {payment_end}")
print(f"30일 이상 관찰 가능한 구매자: {eligible_30d_count:,}명")

결제 데이터 마지막 날짜: 2024-05-08 14:12:45
30일 이상 관찰 가능한 구매자: 59,151명


In [41]:
# 30일 내 재구매 여부
df_repurchase["repurchased_30d"] = (
    df_repurchase["second_purchase"].notna()
    & (
        df_repurchase["second_purchase"]
        <= df_repurchase["first_purchase"] + pd.Timedelta(days=30)
    )
)

In [42]:
# 30일 재구매율
df_eligible_30d = df_repurchase.loc[
    df_repurchase["eligible_30d"]
].copy()

repurchase_30d_rate = (
    df_eligible_30d["repurchased_30d"].mean()
)

print(f"30일 분석 대상자: {len(df_eligible_30d):,}명")
print(f"30일 재구매율: {repurchase_30d_rate:.1%}")
print(f"30일 내 미재구매율: {1 - repurchase_30d_rate:.1%}")

30일 분석 대상자: 59,151명
30일 재구매율: 26.3%
30일 내 미재구매율: 73.7%


In [43]:
# 재구매까지 걸리는 시간
median_days_to_second = (
    df_repurchase.loc[
        df_repurchase["repurchased"],
        "days_to_second"
    ]
    .median()
)

mean_days_to_second = (
    df_repurchase.loc[
        df_repurchase["repurchased"],
        "days_to_second"
    ]
    .mean()
)

print(f"재구매까지 평균 기간: {mean_days_to_second:.1f}일")
print(f"재구매까지 중앙값: {median_days_to_second:.1f}일")

재구매까지 평균 기간: 4.8일
재구매까지 중앙값: 0.3일


#### 최종 비교

In [44]:
# 출력 함수
def pct(x):
    if pd.isna(x):
        return "-"
    return f"{x:.1%}"


def pct_point(x):
    if pd.isna(x):
        return "-"
    return f"{x * 100:.1f}%p"

In [45]:
# 핵심 수치 종합
print("① 친구 연결")
print("친구 0명 비율:", pct(no_friend_rate))
print(
    "친구 수에 따른 참여율 차이:",
    pct_point(friend_participation_gap)
)

print("\n② 질문 참여·콘텐츠")
print("전체 스킵률:", pct(overall_skip_rate))
print(
    "반복 시 스킵률 변화:",
    pct_point(repeat_skip_gap)
)

print("\n③ 투표 수신·반응")
print("투표 미확인율:", pct(1 - vote_read_rate))
print(
    "확인 여부별 상점 방문율 차이:",
    pct_point(vote_shop_gap)
)

print("\n④ 상점·첫 구매")
print(
    "상점 미구매율:",
    pct(shop_no_purchase_rate)
)
print(
    "2회 이상 방문 후 미구매:",
    f"{repeat_no_purchase_count:,}명"
)

print("\n⑤ 구매 후·재구매")
print(
    "30일 미재구매율:",
    pct(1 - repurchase_30d_rate)
)
print(
    "재구매까지 중앙값:",
    f"{median_days_to_second:.1f}일"
)

① 친구 연결
친구 0명 비율: 4.0%
친구 수에 따른 참여율 차이: 0.7%p

② 질문 참여·콘텐츠
전체 스킵률: 100.0%
반복 시 스킵률 변화: 0.0%p

③ 투표 수신·반응
투표 미확인율: 75.7%
확인 여부별 상점 방문율 차이: 7.9%p

④ 상점·첫 구매
상점 미구매율: 90.6%
2회 이상 방문 후 미구매: 3,776명

⑤ 구매 후·재구매
30일 미재구매율: 73.7%
재구매까지 중앙값: 0.3일


In [46]:
# 최종 비교표
df_summary = pd.DataFrame({
    "영역": [
        "① 친구 연결",
        "② 질문 참여·콘텐츠",
        "③ 투표 수신·반응",
        "④ 상점·첫 구매",
        "⑤ 구매 후·재구매"
    ],

    "대표 문제 수치": [
        f"친구 0명 {pct(no_friend_rate)}",
        f"질문 스킵 {pct(overall_skip_rate)}",
        f"투표 미확인 {pct(1 - vote_read_rate)}",
        f"상점 미구매 {pct(shop_no_purchase_rate)}",
        f"30일 미재구매 {pct(1 - repurchase_30d_rate)}"
    ],

    "추가 판단 지표": [
        f"친구 수에 따른 참여율 차이 "
        f"{pct_point(friend_participation_gap)}",

        f"반복 시 스킵률 변화 "
        f"{pct_point(repeat_skip_gap)}",

        f"확인 여부별 상점 방문율 차이 "
        f"{pct_point(vote_shop_gap)}",

        f"2회 이상 방문 후 미구매 "
        f"{repeat_no_purchase_count:,}명",

        f"재구매까지 중앙값 "
        f"{median_days_to_second:.1f}일"
    ]
})

df_summary

,영역,대표 문제 수치,추가 판단 지표
0,① 친구 연결,친구 0명 4.0%,친구 수에 따른 참여율 차이 0.7%p
1,② 질문 참여·콘텐츠,질문 스킵 100.0%,반복 시 스킵률 변화 0.0%p
2,③ 투표 수신·반응,투표 미확인 75.7%,확인 여부별 상점 방문율 차이 7.9%p
3,④ 상점·첫 구매,상점 미구매 90.6%,"2회 이상 방문 후 미구매 3,776명"
4,⑤ 구매 후·재구매,30일 미재구매 73.7%,재구매까지 중앙값 0.3일


#### 문제의 구간 선정을 위한 비교 지표 선정

In [47]:
# 공통 기준 사용자 정의
# 전체 사용자
base_users = set(
    df_user["id"]
    .dropna()
    .astype("int64")
    .unique()
)

total_users = len(base_users)

print(f"전체 사용자 수: {total_users:,}명")

전체 사용자 수: 677,080명


In [48]:
# 공통 지표 계산 함수
def calc_area_metric(area, eligible_users, reached_users):
    eligible_users = set(eligible_users) & base_users
    reached_users = set(reached_users) & eligible_users

    problem_users = eligible_users - reached_users

    return {
        "영역": area,
        "구간 대상자": len(eligible_users),
        "도달 사용자": len(reached_users),
        "구간 도달률": len(reached_users) / len(eligible_users) if eligible_users else np.nan,
        "구간 이탈률": len(problem_users) / len(eligible_users) if eligible_users else np.nan,
        "문제 사용자": len(problem_users),
        "전체 대비 문제 사용자 비율": len(problem_users) / total_users
    }

#### 1. 친구 연결

In [49]:
# 수락된 친구가 있는 사용자
df_friend_accepted = df_friendrequest.loc[
    df_friendrequest["status"] == "A",
    ["send_user_id", "receive_user_id"]
].copy()

friend_users = set(
    pd.concat([
        df_friend_accepted["send_user_id"],
        df_friend_accepted["receive_user_id"]
    ])
    .dropna()
    .astype("int64")
    .unique()
)

In [50]:
# 지표 계산
metric_1 = calc_area_metric(
    "1. 친구 연결",
    base_users,
    friend_users
)

metric_1

{'영역': '1. 친구 연결',
 '구간 대상자': 677080,
 '도달 사용자': 650279,
 '구간 도달률': 0.9604167897441956,
 '구간 이탈률': 0.03958321025580434,
 '문제 사용자': 26801,
 '전체 대비 문제 사용자 비율': 0.03958321025580434}

#### 2. 질문 참여 및 콘텐츠

In [51]:
# 질문 참여 사용자
question_users = set(
    df_userquestionrecord["user_id"]
    .dropna()
    .astype("int64")
    .unique()
)

In [52]:
# 지표 계산
metric_2 = calc_area_metric(
    "2. 질문 참여·콘텐츠",
    friend_users,
    question_users
)

metric_2

{'영역': '2. 질문 참여·콘텐츠',
 '구간 대상자': 650279,
 '도달 사용자': 4830,
 '구간 도달률': 0.007427581084426839,
 '구간 이탈률': 0.9925724189155731,
 '문제 사용자': 645449,
 '전체 대비 문제 사용자 비율': 0.9532832161635257}

#### 3. 투표 수신 및 반응

In [53]:
# 투표 수신 사용자
received_users = set(
    df_userquestionrecord["chosen_user_id"]
    .dropna()
    .astype("int64")
    .unique()
)

In [54]:
# 투표 확인 사용자
read_users = set(
    df_userquestionrecord.loc[
        df_userquestionrecord["has_read"] == 1,
        "chosen_user_id"
    ]
    .dropna()
    .astype("int64")
    .unique()
)

In [55]:
# 지표 계산
metric_3 = calc_area_metric(
    "3. 투표 수신·반응",
    received_users,
    read_users
)

metric_3

{'영역': '3. 투표 수신·반응',
 '구간 대상자': 15426,
 '도달 사용자': 12894,
 '구간 도달률': 0.8358615324776352,
 '구간 이탈률': 0.16413846752236483,
 '문제 사용자': 2532,
 '전체 대비 문제 사용자 비율': 0.0037395876410468484}

#### 4. 상점 및 첫 구매

In [56]:
# 상점 방문 / 구매 이벤트 사용자 연결
df_shop_events = (
    df_hackle_events.loc[
        df_hackle_events["event_key"].isin(
            ["view_shop", "complete_purchase"]
        ),
        ["event_datetime", "event_key", "session_id"]
    ]
    .merge(df_hackle_user, on="session_id", how="inner")
)

In [57]:
# 사용자별 첫 상점 방문 / 첫 구매
first_shop = (
    df_shop_events.loc[df_shop_events["event_key"] == "view_shop"]
    .groupby("user_id")["event_datetime"]
    .min()
)

first_purchase = (
    df_shop_events.loc[df_shop_events["event_key"] == "complete_purchase"]
    .groupby("user_id")["event_datetime"]
    .min()
)

In [58]:
# 첫 상점 방문 이후 구매 여부
df_shop_flow = pd.concat(
    [first_shop.rename("first_shop"),
     first_purchase.rename("first_purchase")],
    axis=1
)

shop_users = set(df_shop_flow["first_shop"].dropna().index)

shop_purchase_users = set(
    df_shop_flow.loc[
        df_shop_flow["first_purchase"].notna()
        & (df_shop_flow["first_purchase"] >= df_shop_flow["first_shop"])
    ].index
)

In [59]:
# 지표 계산
metric_4 = calc_area_metric(
    "4.  상점·첫 구매",
    shop_users,
    shop_purchase_users
)

metric_4

{'영역': '4.  상점·첫 구매',
 '구간 대상자': 17122,
 '도달 사용자': 1617,
 '구간 도달률': 0.09443990188062143,
 '구간 이탈률': 0.9055600981193785,
 '문제 사용자': 15505,
 '전체 대비 문제 사용자 비율': 0.02289980504519407}

#### 5. 구매 후 및 재구매

In [60]:
# 결제 데이터 정리
df_purchase_metric = (
    df_paymenthistory
    .dropna(subset=["user_id", "created_at"])
    .copy()
)

df_purchase_metric["created_at"] = pd.to_datetime(
    df_purchase_metric["created_at"],
    errors="coerce",
    utc=True
).dt.tz_convert(None)

df_purchase_metric = df_purchase_metric.dropna(subset=["created_at"])

In [61]:
# 1시간 기준 구매 세션 생성
df_purchase_metric = df_purchase_metric.sort_values(
    ["user_id", "created_at"]
)

df_purchase_metric["purchase_gap"] = (
    df_purchase_metric
    .groupby("user_id")["created_at"]
    .diff()
)

df_purchase_metric["new_purchase_session"] = (
    df_purchase_metric["purchase_gap"].isna()
    | (df_purchase_metric["purchase_gap"] > pd.Timedelta(hours=1))
)

In [62]:
# 사용자별 구매 세션 수
purchase_session_count = (
    df_purchase_metric
    .groupby("user_id")["new_purchase_session"]
    .sum()
)

purchase_users = set(purchase_session_count.index)

repeat_purchase_users = set(
    purchase_session_count[
        purchase_session_count >= 2
    ].index
)

In [63]:
# 지표 계산
metric_5 = calc_area_metric(
    "5. 구매 후·재구매",
    purchase_users,
    repeat_purchase_users
)

metric_5

{'영역': '5. 구매 후·재구매',
 '구간 대상자': 59192,
 '도달 사용자': 12323,
 '구간 도달률': 0.20818691715096635,
 '구간 이탈률': 0.7918130828490336,
 '문제 사용자': 46869,
 '전체 대비 문제 사용자 비율': 0.06922224847876174}

#### 5개 영역 비교

In [64]:
df_metric_compare = pd.DataFrame([
    metric_1,
    metric_2,
    metric_3,
    metric_4,
    metric_5
])

df_metric_compare

,영역,구간 대상자,도달 사용자,구간 도달률,구간 이탈률,문제 사용자,전체 대비 문제 사용자 비율
0,1. 친구 연결,677080,650279,0.960417,0.039583,26801,0.039583
1,2. 질문 참여·콘텐츠,650279,4830,0.007428,0.992572,645449,0.953283
2,3. 투표 수신·반응,15426,12894,0.835862,0.164138,2532,0.003740
3,4. 상점·첫 구매,17122,1617,0.094440,0.905560,15505,0.022900
4,5. 구매 후·재구매,59192,12323,0.208187,0.791813,46869,0.069222


In [65]:
# 비율로 표시
df_metric_view = df_metric_compare.copy()

for col in [
    "구간 도달률",
    "구간 이탈률",
    "전체 대비 문제 사용자 비율"
]:
    df_metric_view[col] = df_metric_view[col].map(
        lambda x: f"{x:.1%}"
    )

df_metric_view

,영역,구간 대상자,도달 사용자,구간 도달률,구간 이탈률,문제 사용자,전체 대비 문제 사용자 비율
0,1. 친구 연결,677080,650279,96.0%,4.0%,26801,4.0%
1,2. 질문 참여·콘텐츠,650279,4830,0.7%,99.3%,645449,95.3%
2,3. 투표 수신·반응,15426,12894,83.6%,16.4%,2532,0.4%
3,4. 상점·첫 구매,17122,1617,9.4%,90.6%,15505,2.3%
4,5. 구매 후·재구매,59192,12323,20.8%,79.2%,46869,6.9%


In [66]:
coverage = {
    "전체 사용자": df_user["id"].nunique(),
    "친구 요청 사용자": len(friend_users),
    "질문 기록 사용자": df_userquestionrecord["user_id"].nunique(),
    "투표 수신 사용자": df_userquestionrecord["chosen_user_id"].nunique(),
    "Hackle 사용자": df_hackle_user["user_id"].nunique(),
    "결제 사용자": df_paymenthistory["user_id"].nunique()
}

df_coverage = pd.DataFrame(
    coverage.items(),
    columns=["데이터", "사용자 수"]
)

df_coverage["전체 대비 커버리지"] = (
    df_coverage["사용자 수"] / total_users
)

df_coverage

,데이터,사용자 수,전체 대비 커버리지
0,전체 사용자,677080,1.000000
1,친구 요청 사용자,650279,0.960417
2,질문 기록 사용자,4849,0.007162
3,투표 수신 사용자,15426,0.022783
4,Hackle 사용자,226283,0.334204
5,결제 사용자,59192,0.087422


In [67]:
df_coverage_view = df_coverage.copy()

df_coverage_view["전체 대비 커버리지"] = (
    df_coverage_view["전체 대비 커버리지"]
    .map(lambda x: f"{x:.1%}")
)

df_coverage_view

,데이터,사용자 수,전체 대비 커버리지
0,전체 사용자,677080,100.0%
1,친구 요청 사용자,650279,96.0%
2,질문 기록 사용자,4849,0.7%
3,투표 수신 사용자,15426,2.3%
4,Hackle 사용자,226283,33.4%
5,결제 사용자,59192,8.7%


### 전환율 계산하기

#### 파일 불러오기

In [156]:
# preprocessed_2 파일 6개 불러오기
import pandas as pd
import os
import glob

base_path = r"C:\DA15_Part4\sns-project\team4-final-project\data\processed"

files_2 = glob.glob(
    os.path.join(base_path, "*_preprocessed_2.csv")
)

print("파일 수:", len(files_2))

for f in files_2:
    print(os.path.basename(f))

파일 수: 6
accounts_friendrequest_preprocessed_2.csv
accounts_paymenthistory_preprocessed_2.csv
accounts_pointhistory_preprocessed_2.csv
accounts_userquestionrecord_preprocessed_2.csv
accounts_user_preprocessed_2.csv
polls_questionpiece_preprocessed_2.csv


In [157]:
# 필요한 테이블 불러오기
df_user2 = pd.read_csv(
    os.path.join(base_path, "accounts_user_preprocessed_2.csv")
)

df_friend2 = pd.read_csv(
    os.path.join(base_path, "accounts_friendrequest_preprocessed_2.csv")
)

df_uqr2 = pd.read_csv(
    os.path.join(base_path, "accounts_userquestionrecord_preprocessed_2.csv")
)

df_payment2 = pd.read_csv(
    os.path.join(base_path, "accounts_paymenthistory_preprocessed_2.csv")
)

df_point2 = pd.read_csv(
    os.path.join(base_path, "accounts_pointhistory_preprocessed_2.csv")
)

#### ID / 날짜 정리

In [158]:
# 가입
df_user2["id"] = pd.to_numeric(df_user2["id"], errors="coerce")
df_user2["created_at"] = pd.to_datetime(
    df_user2["created_at"], errors="coerce"
)

# 친구
for col in ["send_user_id", "receive_user_id"]:
    df_friend2[col] = pd.to_numeric(
        df_friend2[col], errors="coerce"
    )

df_friend2["created_at"] = pd.to_datetime(
    df_friend2["created_at"], errors="coerce"
)

df_friend2["updated_at"] = pd.to_datetime(
    df_friend2["updated_at"], errors="coerce"
)

# 투표
df_uqr2["user_id"] = pd.to_numeric(
    df_uqr2["user_id"], errors="coerce"
)

df_uqr2["chosen_user_id"] = pd.to_numeric(
    df_uqr2["chosen_user_id"], errors="coerce"
)

df_uqr2["created_at"] = pd.to_datetime(
    df_uqr2["created_at"], errors="coerce"
)

# 결제
df_payment2["user_id"] = pd.to_numeric(
    df_payment2["user_id"], errors="coerce"
)

df_payment2["created_at"] = pd.to_datetime(
    df_payment2["created_at"], errors="coerce"
)

# 재화
df_point2["user_id"] = pd.to_numeric(
    df_point2["user_id"], errors="coerce"
)

df_point2["created_at"] = pd.to_datetime(
    df_point2["created_at"], errors="coerce"
)

#### 1. 가입/진입 사용자

In [165]:
df_join = (
    df_user2[["id", "created_at"]]
    .dropna(subset=["id"])
    .rename(
        columns={
            "id": "user_id",
            "created_at": "join_at"
        }
    )
    .copy()
)

df_join["user_id"] = df_join["user_id"].astype("int64")

all_users = set(df_join["user_id"])

print(f"① 가입/진입: {len(all_users):,}명")

① 가입/진입: 677,075명


#### 2. 친구 관계 형성

In [162]:
df_friend_a = df_friend2[
    df_friend2["status"] == "A"
].copy()

# A 상태가 된 시점은 updated_at 우선 사용
df_friend_a["friend_at"] = (
    df_friend_a["updated_at"]
    .fillna(df_friend_a["created_at"])
)

In [163]:
friend_send = (
    df_friend_a[["send_user_id", "friend_at"]]
    .rename(columns={"send_user_id": "user_id"})
)

friend_receive = (
    df_friend_a[["receive_user_id", "friend_at"]]
    .rename(columns={"receive_user_id": "user_id"})
)

df_friend_users = pd.concat(
    [friend_send, friend_receive],
    ignore_index=True
)

df_friend_users = df_friend_users.dropna(
    subset=["user_id", "friend_at"]
)

df_friend_users["user_id"] = (
    df_friend_users["user_id"].astype("int64")
)

# 사용자별 최초 친구 관계 형성 시점
first_friend = (
    df_friend_users
    .groupby("user_id", as_index=False)["friend_at"]
    .min()
)

In [164]:
stage2 = df_join.merge(
    first_friend,
    on="user_id",
    how="inner"
)

stage2 = stage2[
    stage2["friend_at"] >= stage2["join_at"]
].copy()

friend_users = set(stage2["user_id"])

print(f"② 친구 관계 형성: {len(friend_users):,}명")

② 친구 관계 형성: 574,163명


#### 3. 투표 활동

In [194]:
first_vote = (
    df_uqr2
    .dropna(subset=["chosen_user_id", "created_at"])
    .groupby("chosen_user_id", as_index=False)["created_at"]
    .min()
    .rename(
        columns={
            "chosen_user_id": "user_id",
            "created_at": "vote_at"
        }
    )
)

first_vote["user_id"] = (
    first_vote["user_id"].astype("int64")
)

In [195]:
stage3 = stage2.merge(
    first_vote,
    on="user_id",
    how="inner"
)

vote_users = set(stage3["user_id"])

print(f"③ 투표 수신: {len(vote_users):,}명")

③ 투표 수신: 10,647명


#### 4. 재화 소모

In [235]:
point_spend = (
    df_point2[["user_id"]]
    .dropna()
    .drop_duplicates()
    .copy()
)

point_spend["user_id"] = (
    point_spend["user_id"].astype("int64")
)

In [237]:
stage4 = stage3.merge(
    point_spend,
    on="user_id",
    how="inner"
)

spend_users = set(stage4["user_id"])

print(f"④ 재화 소모: {len(spend_users):,}명")

④ 재화 소모: 3,367명


#### 5. 첫 결제

In [238]:
first_payment = (
    df_payment2
    .dropna(subset=["user_id", "created_at"])
    .groupby("user_id", as_index=False)["created_at"]
    .min()
    .rename(columns={"created_at": "first_payment_at"})
)

first_payment["user_id"] = (
    first_payment["user_id"].astype("int64")
)

In [240]:
stage5 = stage4.merge(
    first_payment,
    on="user_id",
    how="inner"
)

payment_users = set(stage5["user_id"])

print(f"⑤ 첫 결제: {len(payment_users):,}명")

⑤ 첫 결제: 358명


#### 1차 행동 루프 한번에 확인

In [249]:
funnel_1 = pd.DataFrame([
    {
        "퍼널 기준": "가입/진입",
        "사용자 수": len(all_users)
    },
    {
        "퍼널 기준": "친구 관계 형성",
        "사용자 수": len(friend_users)
    },
    {
        "퍼널 기준": "투표 수신",
        "사용자 수": len(vote_users)
    },
    {
        "퍼널 기준": "재화 소모",
        "사용자 수": len(spend_users)
    },
    {
        "퍼널 기준": "첫 결제",
        "사용자 수": len(payment_users)
    }
])

funnel_1["직전 단계 사용자 수"] = (
    funnel_1["사용자 수"].shift(1)
)

funnel_1.loc[0, "직전 단계 사용자 수"] = (
    funnel_1.loc[0, "사용자 수"]
)

funnel_1["구간 전환율"] = (
    funnel_1["사용자 수"]
    / funnel_1["직전 단계 사용자 수"]
)

funnel_1.loc[0, "구간 전환율"] = 1

In [250]:
funnel_1_view = funnel_1.copy()

funnel_1_view["직전 단계 사용자 수"] = (
    funnel_1_view["직전 단계 사용자 수"].astype(int)
)

funnel_1_view["구간 전환율"] = (
    funnel_1_view["구간 전환율"]
    .map(lambda x: f"{x:.1%}")
)

display(funnel_1_view)

,퍼널 기준,사용자 수,직전 단계 사용자 수,구간 전환율
0,가입/진입,677075,677075,100.0%
1,친구 관계 형성,574163,677075,84.8%
2,투표 수신,10647,574163,1.9%
3,재화 소모,3367,10647,31.6%
4,첫 결제,358,3367,10.6%


#### 6. 첫 결제 후 투표 수신

In [241]:
vote_after_payment = (
    df_uqr2[["chosen_user_id", "created_at"]]
    .dropna(subset=["chosen_user_id", "created_at"])
    .rename(columns={
        "chosen_user_id": "user_id",
        "created_at": "vote_after_payment_at"
    })
    .copy()
)

vote_after_payment["user_id"] = (
    vote_after_payment["user_id"].astype("int64")
)

In [242]:
stage6 = (
    stage5[["user_id", "first_payment_at"]]
    .drop_duplicates("user_id")
    .merge(
        vote_after_payment,
        on="user_id",
        how="inner"
    )
)

# 첫 결제 이후 받은 투표만
stage6 = stage6[
    stage6["vote_after_payment_at"] > stage6["first_payment_at"]
].copy()

# 사용자별 첫 번째 재투표 수신
stage6 = (
    stage6
    .sort_values("vote_after_payment_at")
    .drop_duplicates("user_id")
)

revote_users = set(stage6["user_id"])

print(f"⑥ 첫 결제 후 투표 수신: {len(revote_users):,}명")

⑥ 첫 결제 후 투표 수신: 353명


#### 7. 재투표 후 재화소모

In [243]:
point_after_vote = (
    df_point2[["user_id", "created_at"]]
    .dropna(subset=["user_id", "created_at"])
    .rename(columns={
        "created_at": "spend_after_vote_at"
    })
    .copy()
)

point_after_vote["user_id"] = (
    point_after_vote["user_id"].astype("int64")
)

In [244]:
stage7 = stage6.merge(
    point_after_vote,
    on="user_id",
    how="inner"
)

# 재투표 이후 재화 소모만
stage7 = stage7[
    stage7["spend_after_vote_at"]
    > stage7["vote_after_payment_at"]
].copy()

# 사용자별 첫 재화 소모
stage7 = (
    stage7
    .sort_values("spend_after_vote_at")
    .drop_duplicates("user_id")
)

respend_users = set(stage7["user_id"])

print(f"⑦ 재투표 후 재화 소모: {len(respend_users):,}명")

⑦ 재투표 후 재화 소모: 334명


#### 8. 재화 소모 후 재결제

In [245]:
payment_again = (
    df_payment2[["user_id", "created_at"]]
    .dropna(subset=["user_id", "created_at"])
    .rename(columns={
        "created_at": "repurchase_at"
    })
    .copy()
)

payment_again["user_id"] = (
    payment_again["user_id"].astype("int64")
)

In [246]:
stage8 = stage7.merge(
    payment_again,
    on="user_id",
    how="inner"
)

# 재화 소모 이후 다시 결제한 경우만
stage8 = stage8[
    stage8["repurchase_at"]
    > stage8["spend_after_vote_at"]
].copy()

# 사용자별 최초 재결제
stage8 = (
    stage8
    .sort_values("repurchase_at")
    .drop_duplicates("user_id")
)

repurchase_users = set(stage8["user_id"])

print(f"⑧ 재결제: {len(repurchase_users):,}명")

⑧ 재결제: 71명


#### 2차 행동 루프 한번에 확인

In [247]:
funnel_2 = pd.DataFrame([
    {
        "퍼널 기준": "첫 결제",
        "사용자 수": len(payment_users)
    },
    {
        "퍼널 기준": "첫 결제 후 투표 수신",
        "사용자 수": len(revote_users)
    },
    {
        "퍼널 기준": "재투표 후 재화 소모",
        "사용자 수": len(respend_users)
    },
    {
        "퍼널 기준": "재결제",
        "사용자 수": len(repurchase_users)
    }
])

funnel_2["직전 단계 사용자 수"] = (
    funnel_2["사용자 수"].shift(1)
)

funnel_2.loc[0, "직전 단계 사용자 수"] = (
    funnel_2.loc[0, "사용자 수"]
)

funnel_2["구간 전환율"] = (
    funnel_2["사용자 수"]
    / funnel_2["직전 단계 사용자 수"]
)

funnel_2.loc[0, "구간 전환율"] = 1

In [248]:
funnel_2_view = funnel_2.copy()

funnel_2_view["직전 단계 사용자 수"] = (
    funnel_2_view["직전 단계 사용자 수"].astype(int)
)

funnel_2_view["구간 전환율"] = (
    funnel_2_view["구간 전환율"]
    .map(lambda x: f"{x:.1%}")
)

display(funnel_2_view)

,퍼널 기준,사용자 수,직전 단계 사용자 수,구간 전환율
0,첫 결제,358,358,100.0%
1,첫 결제 후 투표 수신,353,358,98.6%
2,재투표 후 재화 소모,334,353,94.6%
3,재결제,71,334,21.3%


#### 전체 행동 루프 한번에 확인

In [251]:
funnel_all = pd.DataFrame([
    {
        "퍼널 기준": "가입/진입",
        "사용자 수": len(all_users)
    },
    {
        "퍼널 기준": "친구 관계 형성",
        "사용자 수": len(friend_users)
    },
    {
        "퍼널 기준": "투표 수신",
        "사용자 수": len(vote_users)
    },
    {
        "퍼널 기준": "재화 소모",
        "사용자 수": len(spend_users)
    },
    {
        "퍼널 기준": "첫 결제",
        "사용자 수": len(payment_users)
    },
    {
        "퍼널 기준": "첫 결제 후 투표 수신",
        "사용자 수": len(revote_users)
    },
    {
        "퍼널 기준": "재투표 후 재화 소모",
        "사용자 수": len(respend_users)
    },
    {
        "퍼널 기준": "재결제",
        "사용자 수": len(repurchase_users)
    }
])

In [252]:
funnel_all["직전 단계 사용자 수"] = (
    funnel_all["사용자 수"].shift(1)
)

funnel_all.loc[0, "직전 단계 사용자 수"] = (
    funnel_all.loc[0, "사용자 수"]
)

funnel_all["구간 전환율"] = (
    funnel_all["사용자 수"]
    / funnel_all["직전 단계 사용자 수"]
)

funnel_all.loc[0, "구간 전환율"] = 1

In [253]:
funnel_all_view = funnel_all.copy()

funnel_all_view["직전 단계 사용자 수"] = (
    funnel_all_view["직전 단계 사용자 수"]
    .astype(int)
)

funnel_all_view["구간 전환율"] = (
    funnel_all_view["구간 전환율"]
    .map(lambda x: f"{x:.1%}")
)

display(funnel_all_view)

,퍼널 기준,사용자 수,직전 단계 사용자 수,구간 전환율
0,가입/진입,677075,677075,100.0%
1,친구 관계 형성,574163,677075,84.8%
2,투표 수신,10647,574163,1.9%
3,재화 소모,3367,10647,31.6%
4,첫 결제,358,3367,10.6%
5,첫 결제 후 투표 수신,353,358,98.6%
6,재투표 후 재화 소모,334,353,94.6%
7,재결제,71,334,21.3%
